## BNN MNIST HLS Testbench

This notebook tests the HLS implementation of the Binarized Neural Network on the Pynq-Z2 board.
It performs the following steps:
1. Loads the Bitstream and configures the DMA.
2. Runs a loop to test variable stream sizes (similar to the C++ testbench).
3. Sends packed data to the FPGA via AXI-Stream.
4. Receives the predicted digit and verifies accuracy.

In [ ]:
from pynq import Overlay
from pynq import allocate
import numpy as np
import time
import random

In [ ]:
ol = Overlay('design_1.bit')
dma = ol.axi_dma_0
dma_send = dma.sendchannel
dma_recv = dma.recvchannel

## Data Pre-processing Helpers

In [ ]:
def quantize_scale(x):
    return 1 if x == -1 else 0

def concat4(li, point):
    result = np.uint32(quantize_scale(li[point]))
    for k in range(1, 32):
        i = point + k
        result <<= 1
        result &= 0xFFFFFFFF
        result |= quantize_scale(li[i])
    return result

def pack_image(flattened_image):
    binarized = np.where(flattened_image > 0, 1, -1)
    padded = np.append(binarized, [1] * 16)
    n = len(padded)
    packed_data = np.zeros(n // 32, dtype=np.uint32)
    for i in range(0, n, 32):
        packed_data[i // 32] = concat4(padded, i)
    return packed_data

In [ ]:
mnist = np.load("dataset/mnist_test_data_original.npy", allow_pickle=True)
X_global = mnist.item().get("data")
y_global = mnist.item().get("label")
X_global = np.reshape(X_global, (10000, 784))
print("Dataset loaded successfully.")

In [ ]:
def run_test(num_samples):
    if num_samples <= 0: return True
    
    print(f"\n-------------------------------------------------")
    print(f"Testing Stream Size: {num_samples}")
    print(f"-------------------------------------------------")
    
    INPUT_PACKED_WIDTH = 25
    
    in_buffer = allocate(shape=(num_samples, INPUT_PACKED_WIDTH), dtype=np.uint32)
    out_buffer = allocate(shape=(num_samples,), dtype=np.int32)
    
    indices = [i % 10000 for i in range(num_samples)]
    
    for i, idx in enumerate(indices):
        packed_img = pack_image(X_global[idx])
        np.copyto(in_buffer[i], packed_img)
        
    dma_send.transfer(in_buffer)
    dma_recv.transfer(out_buffer)
    dma_send.wait()
    dma_recv.wait()
    
    matches = 0
    mismatches = 0
    pass_test = True
    
    for i, idx in enumerate(indices):
        pred = out_buffer[i]
        actual = y_global[idx]
        
        if pred == actual:
            matches += 1
        else:
            mismatches += 1
            # print(f"  Mismatch at index {i}: Pred {pred}, Actual {actual}")

    print(f"Matches: {matches}")
    print(f"Mismatches: {mismatches}")
    
    accuracy = matches / num_samples

    print(f">> Stream Size {num_samples}: Accuracy {accuracy*100:.2f}%")
        
    in_buffer.freebuffer()
    out_buffer.freebuffer()
    
    return accuracy

In [ ]:
test_sizes = [1, 2]
for _ in range(4):
    test_sizes.append(random.randint(3, 1000))
test_sizes.append(1024)

total_errors = 0
for size in test_sizes:
    total_errors += run_test(size)